In [1]:
import os
import pandas as pd
import numpy as np
import re
from tqdm import tqdm

In [2]:
INPUT_BASE = "response_cossim_results"  
OUTPUT_BASE = "response_cossim_stats"

In [3]:
def process_response_stats():
    # 1. Recursively find all response similarity CSVs
    csv_files = []
    if not os.path.exists(INPUT_BASE):
        print(f"Error: Base directory '{INPUT_BASE}' does not exist.")
        return

    for root, dirs, files in os.walk(INPUT_BASE):
        for file in files:
            if file.endswith("_response_cossim.csv"):
                csv_files.append(os.path.join(root, file))

    if not csv_files:
        print(f"No similarity CSVs found in {INPUT_BASE}")
        return

    print(f"Found {len(csv_files)} files. Starting statistical processing...")

    for file_path in tqdm(csv_files, desc="Generating Stats"):
        # Load the similarity data
        df = pd.read_csv(file_path)
        
        # Identify the drift column (usually the first column)
        # Note: Your prompt stats notebook uses 'Drift/Variant'
        drift_col = df.columns[0] 
        prompt_columns = [c for c in df.columns if "_V" in c]
        
        # Group columns by Base ID (e.g., group qna_1_V1, qna_1_V2, qna_1_V3 together)
        base_prompt_map = {}
        for col in prompt_columns:
            base = re.sub(r'_V\d+', '', col)
            if base not in base_prompt_map:
                base_prompt_map[base] = []
            base_prompt_map[base].append(col)
        
        # ==========================================
        # 1. Mean and Std per Base Prompt Group
        # ==========================================
        for base, cols in base_prompt_map.items():
            df[f"{base}_mean"] = df[cols].mean(axis=1)
            df[f"{base}_std"] = df[cols].std(axis=1)
        
        # ==========================================
        # 2. Overall Mean and Std per Drift Level
        # ==========================================
        df["overall_mean"] = df[prompt_columns].mean(axis=1)
        df["overall_std"] = df[prompt_columns].std(axis=1)
        
        # ==========================================
        # 3. Relative Drop % (Baseline = Drift Level 1)
        # ==========================================
        try:
            # Find the overall_mean where Drift equals 1
            baseline_rows = df[df[drift_col] == 1]
            if not baseline_rows.empty:
                baseline_val = baseline_rows["overall_mean"].values[0]
                df["relative_drop_%"] = ((baseline_val - df["overall_mean"]) / baseline_val) * 100
            else:
                df["relative_drop_%"] = 0.0
        except Exception:
            df["relative_drop_%"] = np.nan
        
        # Round for readability
        df = df.round(3)
        
        # ==========================================
        # 4. Save to structured Output Folder
        # ==========================================
        # Reconstruct the relative path to maintain folder hierarchy
        rel_path = os.path.relpath(file_path, INPUT_BASE)
        save_path = os.path.join(OUTPUT_BASE, rel_path.replace(".csv", "_stats.csv"))
        
        os.makedirs(os.path.dirname(save_path), exist_ok=True)
        df.to_csv(save_path, index=False)

    print(f"\nAll stats generated successfully in '{OUTPUT_BASE}'.")

In [4]:
process_response_stats()

Found 36 files. Starting statistical processing...


Generating Stats: 100%|██████████| 36/36 [00:00<00:00, 52.84it/s]


All stats generated successfully in 'response_cossim_stats'.
